In [ ]:
input = {

    "question" : " ", # 질문
    "field" : "", 
    # 특정 분야나 조건으로 제한하는 필터링 요소 시스템 , 사용자 선택으로 결정된다

    "history" : "", # 이전 대화 이력

} #

In [ ]:
 {
      "context": "검색된 문서 문자열", #검색 후 만들어진 참고 문서
      "question": "원래 질문",
      "history": []
  }

 질문
  → 관련 문서 검색
  → 검색 결과(context) + 질문 + 대화 이력
  → 프롬프트
  → LLM
  → 답변

  그리고 문서는 미리 한 번 저장해 둡니다.

  FAQ 문서
  → 임베딩 벡터 생성
  → PGVectorStore 저장

  사용자가 질문할 때마다 FAQ 전체를 읽는 것이 아니라, 질문
  을 벡터로 바꾸고 가장 비슷한 문서만 검색하는 구조입니다.

  ## “이해했다”고 할 수 있는 기준

  다음 5가지를 직접 설명하고 코드로 확인할 수 있으면 이해한
  것입니다.

  1. question, field, history가 왜 필요한가?
  2. retriever는 무엇을 입력받고 무엇을 반환하는가?
  3. RunnableParallel은 왜 사용하는가?
  4. context는 어디서 만들어지고 프롬프트에 어떻게 들어가는
     가?

  5. 검색 결과가 없거나 잘못된 경우 어떻게 되는가?

  특히 이 흐름을 말로 설명할 수 있어야 합니다.

  input_data = {
      "question": "...",
      "field": "...",
      "history": [],
  }

  이 입력이 들어오면:

  {
      "context": "검색된 문서 문자열",
      "question": "원래 질문",
      "history": []
  }

  형태로 바뀌고, 이것이 프롬프트에 들어간 뒤 LLM으로 전달됩
  니다.

  즉 RunnableParallel은 입력을 여러 갈래로 동시에 처리해서
  프롬프트에 필요한 재료를 모으는 역할입니다.

  ## 먼저 LangChain 없이 만들어보세요

  처음부터 PostgreSQL, 임베딩, OpenAI를 붙이면 어디서 문제
  가 생겼는지 알기 어렵습니다. 먼저 검색을 단순한 문자열 포
  함 검색으로 흉내 내면 됩니다.

  documents = [
      {
          "field": "의료 분야",
          "text": "CCTV 영상 판독은 전문의의 확인이 필요합
          니다."
      },
      {
          "field": "법률 분야",
          "text": "개인정보가 포함된 영상은 보관과 이용에
          주의해야 합니다."
      },
  ]

  검색 함수부터 만듭니다.

  def retrieve(question, field):
      results = []

      for document in documents:
          if document["field"] == field and any(
              word in document["text"]
              for word in question.split()
          ):
              results.append(document["text"])

      return results

  문서 포맷팅 함수를 만듭니다.

  def format_docs(docs):
      return "\n\n".join(docs)

  그다음 RAG 전체 흐름을 직접 작성합니다.

  def rag_chain(input_data):
      question = input_data["question"]
      field = input_data["field"]
      history = input_data["history"]

      docs = retrieve(question, field)
      context = format_docs(docs)

      prompt = f"""
  너는 질문에 답하는 도우미다.

  대화 이력:
  {history}

  참고 문서:
  {context}

  질문:
  {question}

  참고 문서에 근거해서 답변하라.
  """

      return {
          "prompt": prompt,
          "context": context,
          "question": question,
          "history": history,
      }

  실행해보면 됩니다.

  result = rag_chain({
      "question": "CCTV 영상을 어떻게 확인해야 하나요?",
      "field": "의료 분야",
      "history": [],
  })

  print(result["context"])
  print(result["prompt"])

  이 코드에서 다이어그램의 각 요소를 대응시키면 다음과 같습
  니다.

   다이어그램            직접 만든 코드
  ━━━━━━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   input_data            함수 입력값
  ────────────────────  ───────────────────────────────────
   retriever             retrieve()
  ────────────────────  ───────────────────────────────────
   format_docs           format_docs()
  ────────────────────  ───────────────────────────────────
   context               검색된 문서 문자열
  ────────────────────  ───────────────────────────────────
   rag_history_prompt    prompt 문자열
  ────────────────────  ───────────────────────────────────
   LLM                   아직 없는 상태
  ────────────────────  ───────────────────────────────────
   StrOutputParser       LLM 결과를 문자열로 변환하는 단계

  ## 그다음 가짜 LLM을 붙입니다

  def fake_llm(prompt):
      if "참고 문서:" not in prompt:
          return "답변할 수 없습니다."

      return "참고 문서를 바탕으로 답변했습니다."

  def rag_chain(input_data):
      question = input_data["question"]
      field = input_data["field"]
      history = input_data["history"]

      docs = retrieve(question, field)
      context = format_docs(docs)

      prompt = f"""
  대화 이력:
  {history}

  참고 문서:
  {context}

  질문:
  {question}
  """

      answer = fake_llm(prompt)

      return answer

  이제 다음 질문에 답할 수 있어야 합니다.

  - retrieve()를 삭제하면 어떤 문제가 생기는가?
  - context를 프롬프트에서 삭제하면 어떤 문제가 생기는가?
  - history를 추가하면 무엇이 좋아지는가?
  - 검색 문서가 비어 있으면 LLM은 무엇을 해야 하는가?
  - field 필터를 제거하면 어떤 문서가 섞일 수 있는가?

  ## 마지막에 LangChain으로 바꾸세요

  직접 만든 함수 구조를 LangChain 표현으로 옮기면 됩니다.

  retriever_chain = (
      {
          "context": retriever_chain,
          "question": itemgetter("question"),
          "history": itemgetter("history"),
      }
      | rag_history_prompt
      | llm
      | StrOutputParser()
  )

  여기서 중요한 것은 LangChain 문법을 외우는 것이 아닙니다.

  입력 dict
  → 각 값 추출
  → 검색 결과 생성
  → prompt 변수에 주입
  1. 문서 2개를 리스트로 직접 만든다.
  2. 문자열 검색 retriever를 만든다.
  3. context를 출력한다.
  4. question, context, history로 prompt를 만든다.
  7. 그다음 LangChain RunnableParallel로 옮긴다.
  8. 마지막에 임베딩과 PGVector를 붙인다.

  처음부터 PGVector와 실제 LLM까지 연결하지 말고, 각 단계의
  입력과 출력을 print()로 확인하는 게 좋습니다.

  최종적으로 아래처럼 설명할 수 있으면 충분히 이해한 것입니
  다.

  > 사용자의 입력은 질문, 분야, 대화 이력으로 구성된다. 질
  > 문과 분야를 retriever에 전달해 관련 문서를 검색하고, 검
  > 색 문서를 context 문자열로 변환한다. RunnableParallel은
  > context, question, history를 동시에 준비한다. 이 값들이
  > prompt에 들어가고, LLM이 답변을 생성한다. 마지막으로
  > StrOutputParser가 결과를 문자열로 변환한다.

  지금 상태에서는 “개념을 더 읽어야 하는 단계”라기보다, 작
  은 가짜 RAG를 직접 작성하면서 각 변수의 흐름을 확인해야
  하는 단계에 가깝습니다.